# Tests: `fasterai.misc.fc_decomposer` (source `nbs/misc/fc_decomposer.ipynb`)

In [ ]:
from fastcore.test import *
import torch
import torch.nn as nn
from fasterai.misc.fc_decomposer import *

In [ ]:
from fastcore.test import *

# SVD decomposition preserves output approximately
model = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 10))
x = torch.randn(4, 32)
out_orig = model(x)

decomposer = FC_Decomposer()
model_dec = decomposer.decompose(model, percent_removed=0.5)
out_dec = model_dec(x)
test_close(out_orig, out_dec, eps=1.0)

# Decomposed structure: Linear → Sequential(Linear, Linear)
assert isinstance(model_dec[0], nn.Sequential)
assert len(model_dec[0]) == 2

# percent_removed=0 → very close output
m2 = nn.Sequential(nn.Linear(32, 64))
x2 = torch.randn(4, 32)
out2 = m2(x2)
m2_dec = decomposer.decompose(m2, percent_removed=0.0)
test_close(out2, m2_dec(x2), eps=1e-4)

# L >= 1 always
m3 = nn.Sequential(nn.Linear(10, 20))
m3_dec = decomposer.decompose(m3, percent_removed=0.95)
assert m3_dec[0][0].out_features >= 1

# Invalid percent_removed raises ValueError
with ExceptionExpected(ValueError):
    decomposer.decompose(nn.Sequential(nn.Linear(10, 10)), percent_removed=1.0)

# --- energy_threshold ---
m4 = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 10))
m4_99 = decomposer.decompose(m4, energy_threshold=0.99)
m4_50 = decomposer.decompose(m4, percent_removed=0.5)
assert m4_99[0][0].out_features >= m4_50[0][0].out_features

# --- layers / exclude ---
m6 = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 10))
m6_sel = decomposer.decompose(m6, 0.5, layers=['0'])
assert isinstance(m6_sel[0], nn.Sequential)
assert isinstance(m6_sel[2], nn.Linear)

m7 = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 10))
m7_exc = decomposer.decompose(m7, 0.5, exclude=['2'])
assert isinstance(m7_exc[0], nn.Sequential)
assert isinstance(m7_exc[2], nn.Linear)

# --- ASVD: activation-aware SVD ---
m8 = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 10))
x8 = torch.randn(16, 32)
out8 = m8(x8)

# ASVD with calibration data
m8_asvd = decomposer.decompose(m8, 0.5, data=[x8])
out8_asvd = m8_asvd(x8)

# Standard SVD for comparison
m8_svd = decomposer.decompose(m8, 0.5)
out8_svd = m8_svd(x8)

# Both produce valid outputs
assert torch.isfinite(out8_asvd).all()
assert torch.isfinite(out8_svd).all()

# ASVD should have lower reconstruction error on the calibration data
err_asvd = (out8 - out8_asvd).pow(2).mean()
err_svd = (out8 - out8_svd).pow(2).mean()
# Note: on random weights this may not always hold, but scaling should not make things worse
assert torch.isfinite(err_asvd)

# ASVD with data=None → same as standard SVD
m9 = nn.Sequential(nn.Linear(10, 20))
m9_no_data = decomposer.decompose(m9, 0.5, data=None)
assert isinstance(m9_no_data[0], nn.Sequential)